In [5]:
# =========================================================================
# Overhead Satellite Collection Forecast
# Dependencies: pip install skyfield sgp4 jplephem
# =========================================================================

import os
from datetime import timedelta
from skyfield.api import load, wgs84

# 1. Timescale and ephemeris setup
ts = load.timescale()
t0 = ts.now()
t1 = ts.utc(t0.utc_datetime() + timedelta(hours=24))

# Load JPL ephemeris for solar illumination math (Skyfield auto-downloads de421.bsp if missing)
eph = load('de421.bsp')
sun = eph['sun']
earth = eph['earth']

# Relative path ensures portability across any machine
file_path = 'active_sats.txt'

if not os.path.exists(file_path):
    print(f"ERROR: '{file_path}' not found in working directory.")
    print("Download active TLE file from CelesTrak and save it as 'active_sats.txt'.")
else:
    satellites = load.tle_file(file_path)
    
    if len(satellites) == 0:
        print(f"ERROR: '{file_path}' loaded 0 satellites. Check file formatting.")
    else:
        # Calculate the age of the orbital data to warn of positional drift
        sample_sat = satellites[0]
        data_age_days = t0 - sample_sat.epoch
        
        print("=" * 73)
        print("                       OVERHEAD COLLECTION FORECAST                      ")
        print("=" * 73)
        if abs(data_age_days) > 3.0:
            print(f"[WARNING] TLE Data is {abs(data_age_days):.1f} days old. Atmospheric drag drift expected.")
            print("[ACTION]  Download fresh active_sats.txt before field deployment.")
        else:
            print(f"[OK] TLE Data is {abs(data_age_days):.1f} days old. Prediction confidence high.")
        print("=" * 73 + "\n")

        # Define target constellations
        optical_constellations = ['GAOFEN', 'JILIN']
        sar_identifiers = [
            'YAOGAN-29', 'YAOGAN-33', 
            'YAOGAN-1 ', 'YAOGAN-3 ', 'YAOGAN-10', 'YAOGAN-13', 'YAOGAN-18'
        ]
        
        optical_sats = [sat for sat in satellites if any(name in sat.name for name in optical_constellations)]
        sar_sats = [sat for sat in satellites if any(name in sat.name for name in sar_identifiers)]
        
        # Coordinates input (Default: Tuscaloosa, AL - adjust lat/lon as needed)
        ground_station = wgs84.latlon(33.1893, -87.5652)
        observer = earth + ground_station  # Topocentric observer for solar elevation math
        
        overhead_intervals = []
        
        # Optical processing (Daylight / twilight only)
        for sat in optical_sats:
            times, events = sat.find_events(ground_station, t0, t1, altitude_degrees=45.0)
            rise_time = None
            for ti, event in zip(times, events):
                if event == 0:  # Rise above 45 deg
                    rise_time = ti.tt
                elif event == 2:  # Set below 45 deg
                    if rise_time is not None:
                        # Midpoint of the pass
                        mid_tt = (rise_time + ti.tt) / 2.0
                        mid_time = ts.tt(jd=mid_tt)
                        
                        # Sun altitude check at ground coordinates
                        astrometric = observer.at(mid_time).observe(sun)
                        alt, az, distance = astrometric.apparent().altaz()
                        
                        # Sun > -6 deg defines civil twilight to daylight collection window
                        if alt.degrees > -6.0:
                            overhead_intervals.append([rise_time, ti.tt])
                        rise_time = None
                        
        # SAR processing (Day and night collection capable)
        for sat in sar_sats:
            times, events = sat.find_events(ground_station, t0, t1, altitude_degrees=20.0)
            rise_time = None
            for ti, event in zip(times, events):
                if event == 0:  # Rise above 20 deg
                    rise_time = ti.tt
                elif event == 2:  # Set below 20 deg
                    if rise_time is not None:
                        overhead_intervals.append([rise_time, ti.tt])
                        rise_time = None
                        
        # Sort and merge overlapping collection intervals
        overhead_intervals.sort(key=lambda x: x[0])
        merged_threats = []
        for interval in overhead_intervals:
            if not merged_threats or merged_threats[-1][1] < interval[0]:
                merged_threats.append(interval)
            else:
                merged_threats[-1][1] = max(merged_threats[-1][1], interval[1])
                
        # Build timeline
        timeline = []
        current_time = t0.tt
        end_time = t1.tt
        
        for threat_start, threat_end in merged_threats:
            if current_time < threat_start:
                timeline.append((current_time, threat_start, "CLEAR"))
            timeline.append((threat_start, threat_end, "THREAT"))
            current_time = max(current_time, threat_end)
            
        if current_time < end_time:
            timeline.append((current_time, end_time, "CLEAR"))
            
        # Schedule output
        print("TIME (UTC)                   | STATUS              | DURATION")
        print("-" * 68)
        
        for start_jd, end_jd, status in timeline:
            start_obj = ts.tt(jd=start_jd)
            end_obj = ts.tt(jd=end_jd)
            duration_minutes = (end_jd - start_jd) * 24 * 60
            
            start_str = start_obj.utc_strftime('%m-%d %H:%M')
            end_str = end_obj.utc_strftime('%m-%d %H:%M')
            
            if status == "THREAT":
                print(f"{start_str} to {end_str} | [X] SAT OVERHEAD  | {duration_minutes:5.1f} min")
            elif status == "CLEAR":
                if duration_minutes < 15.0:
                    print(f"{start_str} to {end_str} | [-] MICRO-GAP     | {duration_minutes:5.1f} min (Too short)")
                else:
                    print(f"{start_str} to {end_str} | [O] ALL CLEAR     | {duration_minutes:5.1f} min")

                       OVERHEAD COLLECTION FORECAST                      
[OK] TLE Data is 0.4 days old. Prediction confidence high.

TIME (UTC)                   | STATUS              | DURATION
--------------------------------------------------------------------
09-28 20:54 to 09-28 21:03 | [-] MICRO-GAP     |   8.9 min (Too short)
09-28 21:03 to 09-28 21:05 | [X] SAT OVERHEAD  |   1.3 min
09-28 21:05 to 09-28 21:09 | [-] MICRO-GAP     |   4.3 min (Too short)
09-28 21:09 to 09-28 21:10 | [X] SAT OVERHEAD  |   1.2 min
09-28 21:10 to 09-28 22:04 | [O] ALL CLEAR     |  54.2 min
09-28 22:04 to 09-28 22:06 | [X] SAT OVERHEAD  |   1.7 min
09-28 22:06 to 09-28 23:10 | [O] ALL CLEAR     |  63.9 min
09-28 23:10 to 09-28 23:12 | [X] SAT OVERHEAD  |   2.4 min
09-28 23:12 to 09-28 23:13 | [-] MICRO-GAP     |   0.1 min (Too short)
09-28 23:13 to 09-28 23:14 | [X] SAT OVERHEAD  |   1.1 min
09-28 23:14 to 09-28 23:14 | [-] MICRO-GAP     |   0.3 min (Too short)
09-28 23:14 to 09-28 23:16 | [X] SAT O